# 09 · Functions

**TL;DR** — a function is a **named, reusable block** with its own private workspace.
Define once, call many times.

```
           arguments
               │
     ┌─────────▼─────────┐
     │   def f(params):  │      benefits:
     │       body        │      · modularity
     │       return x    │      · readability
     └─────────┬─────────┘      · reusability
               │
           return value
```

**Agenda**: def & docstrings → parameters vs arguments → argument kinds →
`*args`/`**kwargs` → return → memory & call stack → scope (LEGB) →
global/nonlocal → nested functions → first-class functions → recursion → closures

## 1. Defining — `def`, docstring, `return`

In [1]:
def is_even(number):
    """Return True if number is divisible by 2, else False."""
    return number % 2 == 0

for i in range(1, 6):
    print(i, is_even(i))

1 False
2 True
3 False
4 True
5 False


In [2]:
# the docstring travels with the function
print(is_even.__doc__)

# built-ins have docstrings too
print(print.__doc__[:60], '...')

Return True if number is divisible by 2, else False.
Prints the values to a stream, or to sys.stdout by default.
 ...


In [3]:
# type hints (annotations) — documentation for humans & tools;
# Python does NOT enforce them at runtime
def area(width: float, height: float = 1.0) -> float:
    return width * height

print(area(3.5, 2))
print(area.__annotations__)
print(area('ab', 3))     # runs anyway — hints are not checks!

7.0
{'width': <class 'float'>, 'height': <class 'float'>, 'return': <class 'float'>}
ababab


### Two points of view
- **Creator**: writes the body once, documents it
- **User**: calls it, reads only the docstring — never needs the body

### Parameters vs Arguments

| Term | Where | Example |
|---|---|---|
| **Parameter** | in the `def` — placeholder | `def power(a, b)` → `a`, `b` |
| **Argument** | in the call — actual value | `power(2, 3)` → `2`, `3` |

## 2. Argument Kinds

In [4]:
def power(a=1, b=1):
    """a raised to b; both default to 1."""
    return a ** b

# positional — matched by ORDER
print(power(2, 3))

# keyword — matched by NAME, order free
print(power(b=3, a=2))

# default — missing args fall back
print(power(2))
print(power())

8
8
2
1


In [5]:
# rule: positional args must come BEFORE keyword args in a call
print(power(2, b=3))    # ✅

8


### ⚠️ The mutable default argument trap

Defaults are evaluated **once, at def time** — a mutable default is
**shared across calls**.

In [6]:
def buggy(item, basket=[]):        # ❌ one list, created once
    basket.append(item)
    return basket

print(buggy('apple'))
print(buggy('mango'))              # surprise: apple is still there!

['apple']
['apple', 'mango']


In [7]:
def fixed(item, basket=None):      # ✅ the standard fix
    if basket is None:
        basket = []                # fresh list per call
    basket.append(item)
    return basket

print(fixed('apple'))
print(fixed('mango'))

['apple']
['mango']


In [8]:
# ❌ positional after keyword → SyntaxError
print(power(a=2, 3))

SyntaxError: positional argument follows keyword argument (63219708.py, line 2)

### `*args` — any number of positional arguments (arrives as a tuple)

In [9]:
def multiply(*args):
    print('args is', args, type(args))
    product = 1
    for value in args:
        product *= value
    return product

print(multiply(1, 2, 3, 4))
print(multiply(10))

args is (1, 2, 3, 4) <class 'tuple'>
24
args is (10,) <class 'tuple'>
10


### `**kwargs` — any number of keyword arguments (arrives as a dict)

In [10]:
def display(**kwargs):
    print('kwargs is', kwargs, type(kwargs))
    for key, value in kwargs.items():
        print(key, '→', value)

display(india='delhi', japan='tokyo')

kwargs is {'india': 'delhi', 'japan': 'tokyo'} <class 'dict'>
india → delhi
japan → tokyo


In [11]:
# ordering rule in the def:  normal → *args → **kwargs
def profile(name, *scores, **details):
    print(name, scores, details)

profile('lakshay', 90, 85, 77, city='delhi', age=25)

# names 'args'/'kwargs' are convention only — the stars do the work

lakshay (90, 85, 77) {'city': 'delhi', 'age': 25}


In [12]:
# the stars work at the CALL SITE too — unpack containers into arguments
def power(a, b):
    return a ** b

pair = [2, 10]
print(power(*pair))            # power(2, 10)

named = {'a': 2, 'b': 5}
print(power(**named))          # power(a=2, b=5)

1024
32


### Keyword-only (`*`) and positional-only (`/`) parameters

```
 def f(pos_only, /, normal, *, kw_only)
           │                      │
  caller MUST use position   caller MUST use name
```

In [13]:
def divide(a, b, /, *, precision=2):
    return round(a / b, precision)

print(divide(10, 3))
print(divide(10, 3, precision=4))

3.33
3.3333


In [14]:
# ❌ positional-only passed by name → TypeError
divide(a=10, b=3)

TypeError: divide() got some positional-only arguments passed as keyword arguments: 'a, b'

## 3. Return — every function returns *something*

No `return` (or bare `return`) → `None`.

In [15]:
def greet(name):
    print('hello', name)      # prints, but returns nothing

result = greet('lakshay')
print(result)                 # None

hello lakshay
None


In [16]:
# classic trap: list.append returns None too
L = [1, 2, 3]
print(L.append(4))    # None — append mutates, doesn't return
print(L)

None
[1, 2, 3, 4]


In [17]:
# returning multiple values → really ONE tuple, unpacked
def min_max(numbers):
    return min(numbers), max(numbers)

lo, hi = min_max([3, 1, 4, 1, 5])
print(lo, hi)

1 5


## 4. How Functions Execute in Memory — the call stack

Each call pushes a **frame** (its own local variables); `return` pops it.

```
  def g(): return 2
  def f(): return g() + 1
  f()

  step 1        step 2        step 3        step 4
  ┌──────┐      ┌──────┐      ┌──────┐      ┌──────┐
  │      │      │ g()  │◄push │      │◄pop  │      │
  │ f()  │◄push │ f()  │      │ f()  │      │      │◄pop
  ├──────┤      ├──────┤      ├──────┤      ├──────┤
  │ main │      │ main │      │ main │      │ main │
  └──────┘      └──────┘      └──────┘      └──────┘
                             g returns 2   f returns 3
```

In [18]:
def g():
    print('  g: pushed')
    return 2

def f():
    print(' f: pushed, calling g...')
    value = g() + 1
    print(' f: g popped, resuming')
    return value

print('main: calling f...')
print('main: result =', f())

main: calling f...
 f: pushed, calling g...
  g: pushed
 f: g popped, resuming
main: result = 3


## 5. Variable Scope — the LEGB rule

Name lookup order, inside-out:

```
  ┌─────────────────────────── B  built-in (print, len...) ─┐
  │  ┌──────────────────────── G  global (module top level) │
  │  │  ┌───────────────────── E  enclosing (outer def)     │
  │  │  │  ┌────────────────── L  local (current def)       │
  │  │  │  │   x?  ── search L → E → G → B                  │
  │  │  │  └─────────────────────────────────────────────   │
```

In [19]:
# reading a global inside a function — fine
x = 10

def read_global():
    print('sees global x =', x)

read_global()

sees global x = 10


In [20]:
# a local with the same name SHADOWS the global
x = 10

def shadow():
    x = 99            # new local x
    print('local x =', x)

shadow()
print('global x still', x)

local x = 99
global x still 10


In [21]:
# ❌ read-then-assign without declaring → UnboundLocalError
x = 10

def broken():
    x += 1      # assignment makes x local, but it's read before set
    return x

broken()

UnboundLocalError: cannot access local variable 'x' where it is not associated with a value

In [22]:
# fix 1: global — rebind the module-level name (use sparingly!)
x = 10

def fixed():
    global x
    x += 1

fixed()
print(x)    # 11

11


In [23]:
# fix 2 (better): pass in, return out — no hidden coupling
x = 10

def pure(value):
    return value + 1

x = pure(x)
print(x)

11


In [24]:
# functions can't see each other's locals
def first():
    secret = 42

def second():
    # print(secret)  # would raise NameError
    pass

first(); second()
print('each call has its own private frame')

each call has its own private frame


## 6. Nested Functions

A `def` inside a `def`. The inner one is invisible outside.

In [25]:
def outer():
    def inner():
        print('inner runs')
    print('outer runs')
    inner()

outer()

outer runs
inner runs


In [26]:
# ❌ inner is local to outer → NameError
def outer2():
    def inner2():
        pass

outer2()
inner2()

NameError: name 'inner2' is not defined

In [27]:
# nonlocal — assign to the ENCLOSING function's variable
def counter_demo():
    count = 0
    def bump():
        nonlocal count      # without this: UnboundLocalError
        count += 1
    bump(); bump(); bump()
    return count

print(counter_demo())

3


## 7. Functions are First-Class Citizens

A function is an object like any other: it has a type and an id, and can be
renamed, deleted, stored, passed, and returned.

In [28]:
def square(num):
    return num ** 2

print(type(square))
print(id(square))

<class 'function'>
4448529824


In [29]:
# 1) rename — a second label on the same object
s = square
print(s(4))

16


In [30]:
# 2) delete the original name — the object lives while any name remains
del square
print(s(5))            # still works via s

25


In [31]:
# 3) store in containers
def square(num):
    return num ** 2

toolbox = [square, len, max]
print(toolbox[0](6))

unique_funcs = {square, len}      # functions are hashable
print(len(unique_funcs))

36
2


In [32]:
# 4) return a function from a function
def make_power(exponent):
    def raiser(base):
        return base ** exponent
    return raiser

cube = make_power(3)
print(cube(2))
print(make_power(2)(9))    # call the returned function immediately

8
81


In [33]:
# 5) pass a function as an argument
def shout(text):
    return text.upper()

def whisper(text):
    return text.lower()

def speak(style, text):
    print(style(text))

speak(shout, 'Hello There')
speak(whisper, 'Hello There')

HELLO THERE
hello there


## 8. Recursion — a function calling itself

Two mandatory parts: **base case** (stop) + **recursive case** (shrink the problem).

```
 factorial(4)                    call stack grows, then unwinds:
 = 4 * factorial(3)                  fact(1) = 1        ▲ returns
 = 4 * 3 * factorial(2)             fact(2) = 2*1       │
 = 4 * 3 * 2 * factorial(1)        fact(3) = 3*2        │
 = 4 * 3 * 2 * 1 = 24             fact(4) = 4*6         │
```

In [34]:
def factorial(n):
    if n <= 1:            # base case — MUST exist
        return 1
    return n * factorial(n - 1)

print(factorial(4))
print(factorial(10))

24
3628800


In [35]:
# fibonacci: 0 1 1 2 3 5 8 ...
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print([fib(i) for i in range(10)])

[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]


In [36]:
# MUTUAL recursion — two functions calling each other (still needs a base case)
def is_even_r(n):
    return True if n == 0 else is_odd_r(n - 1)

def is_odd_r(n):
    return False if n == 0 else is_even_r(n - 1)

print(is_even_r(10), is_odd_r(7))

True True


In [37]:
# ❌ no base case → RecursionError (Python caps stack depth, default ~1000)
import sys
sys.setrecursionlimit(50)      # shrink the cap so the traceback stays short

def endless():
    return endless()

endless()

RecursionError: maximum recursion depth exceeded

In [38]:
sys.setrecursionlimit(1000)    # restore the default cap

## 9. Closures — a function that remembers its birthplace

The inner function keeps the enclosing variables alive **after the outer
function has returned**.

In [39]:
def make_counter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

clicks = make_counter()
print(clicks(), clicks(), clicks())    # 1 2 3 — state survives between calls

other = make_counter()
print(other())                         # 1 — each closure has its OWN count

1 2 3
1


In [40]:
# peek inside: the captured variable lives in __closure__
print(clicks.__closure__[0].cell_contents)

3


---
## Recap

| Concept | One-liner |
|---|---|
| parameter vs argument | in the def vs in the call |
| `*args` / `**kwargs` | tuple / dict; order: normal → * → ** |
| `/` and `*` in def | positional-only / keyword-only |
| no `return` | function yields `None` |
| call stack | each call = frame pushed, return = popped |
| LEGB | local → enclosing → global → built-in |
| `global` / `nonlocal` | rebind outer names (use sparingly) |
| first-class | rename, store, pass, return functions |
| recursion | base case + shrinking case |
| closure | inner function keeps outer variables alive |